<a href="https://colab.research.google.com/github/JinyongPark72/Login/blob/master/drive_cb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip3 install openai-whisper

In [ ]:
import whisper
import re
from openai import OpenAI
import json
import os

# 🔑 여기에 본인 API 키 넣기
import os
os.environ["OPENAI_API_KEY"] = "api_key"

# 1. 음성 → 텍스트
def speech_to_text(audio_file):
    model = whisper.load_model("small")
    result = model.transcribe(audio_file, language='ko')
    return result["text"]


# 2. 텍스트 → 출발/도착 추출 (AI 방식)
client = OpenAI()

def extract_with_ai(text):
    prompt = f"""
    사용자의 문장에서 출발지와 도착지를 추출해서 JSON으로 만들어줘.

    문장: "{text}"

    형식:
    {{
        "출발지": "",
        "도착지": ""
    }}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    content = response.choices[0].message.content

    # 🔥 JSON 안정화 처리
    content = content.strip().replace("```json", "").replace("```", "")

    try:
        data = json.loads(content)
        return data.get("출발지"), data.get("도착지")
    except:
        return None, None


# 3. 기존 방식 (fallback용)
def clean_location(text):
    particles = ["에서", "에", "로", "으로"]
    for p in particles:
        if text.endswith(p):
            text = text[:-len(p)]
    return text.strip()


def extract_locations(text):
    match = re.search(r"(.+?)에서 (.+)", text)
    if match:
        start = clean_location(match.group(1))
        end = clean_location(match.group(2))
        return start, end

    parts = text.split()
    if len(parts) >= 2:
        start = clean_location(parts[0])
        end = clean_location(parts[-1])
        return start, end

    return None, None


# 4. 실행
if __name__ == "__main__":
    audio_file = "test.m4a"

    text = speech_to_text(audio_file)
    print(f"인식된 문장: {text}")

    # 🔥 먼저 AI 시도
    start, end = extract_with_ai(text)

    # ❗ 실패하면 기존 방식 사용
    if not start or not end:
        print("AI 분석 실패 → 기존 방식 시도")
        start, end = extract_locations(text)

    if start and end:
        print(f"출발지: {start}")
        print(f"도착지: {end}")
    else:
        print("❌ 위치 추출 실패 (다시 말해 주세요)")

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


인식된 문장:  강남에 서 잠실
출발지: 강남
도착지: 잠실
